In [ ]:
!pip install -q sentence-transformers scikit-learn pandas torch


In [ ]:
import pandas as pd
from sentence_transformers import CrossEncoder


In [ ]:
from datasets import load_dataset

dataset = load_dataset("microsoft/ms_marco", "v1.1")


README.md: 0.00B [00:00, ?B/s]

v1.1/validation-00000-of-00001.parquet:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

v1.1/train-00000-of-00001.parquet:   0%|          | 0.00/175M [00:00<?, ?B/s]

v1.1/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

Generating validation split:   0%|          | 0/10047 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/82326 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/9650 [00:00<?, ? examples/s]

In [ ]:
print(dataset)


DatasetDict({
    validation: Dataset({
        features: ['answers', 'passages', 'query', 'query_id', 'query_type', 'wellFormedAnswers'],
        num_rows: 10047
    })
    train: Dataset({
        features: ['answers', 'passages', 'query', 'query_id', 'query_type', 'wellFormedAnswers'],
        num_rows: 82326
    })
    test: Dataset({
        features: ['answers', 'passages', 'query', 'query_id', 'query_type', 'wellFormedAnswers'],
        num_rows: 9650
    })
})


In [ ]:
print(dataset["train"][0])


{'answers': ['Results-Based Accountability is a disciplined way of thinking and taking action that communities can use to improve the lives of children, youth, families, adults and the community as a whole.'], 'passages': {'is_selected': [0, 0, 0, 0, 0, 1, 0, 0, 0, 0], 'passage_text': ["Since 2007, the RBA's outstanding reputation has been affected by the 'Securency' or NPA scandal. These RBA subsidiaries were involved in bribing overseas officials so that Australia might win lucrative note-printing contracts. The assets of the bank include the gold and foreign exchange reserves of Australia, which is estimated to have a net worth of A$101 billion. Nearly 94% of the RBA's employees work at its headquarters in Sydney, New South Wales and at the Business Resumption Site.", "The Reserve Bank of Australia (RBA) came into being on 14 January 1960 as Australia 's central bank and banknote issuing authority, when the Reserve Bank Act 1959 removed the central banking functions from the Commonw

In [ ]:
dataset["train"][0]["passages"]


{'is_selected': [0, 0, 0, 0, 0, 1, 0, 0, 0, 0],
 'passage_text': ["Since 2007, the RBA's outstanding reputation has been affected by the 'Securency' or NPA scandal. These RBA subsidiaries were involved in bribing overseas officials so that Australia might win lucrative note-printing contracts. The assets of the bank include the gold and foreign exchange reserves of Australia, which is estimated to have a net worth of A$101 billion. Nearly 94% of the RBA's employees work at its headquarters in Sydney, New South Wales and at the Business Resumption Site.",
  "The Reserve Bank of Australia (RBA) came into being on 14 January 1960 as Australia 's central bank and banknote issuing authority, when the Reserve Bank Act 1959 removed the central banking functions from the Commonwealth Bank. The assets of the bank include the gold and foreign exchange reserves of Australia, which is estimated to have a net worth of A$101 billion. Nearly 94% of the RBA's employees work at its headquarters in Sydn

In [ ]:
rows = []

for item in dataset["train"]:
    query = item["query"]
    answers = item["answers"]

    for p in item["passages"]["passage_text"]:
        rows.append({
            "query": query,
            "finalpassage": p,
            "answers": answers
        })


In [ ]:
import pandas as pd

df = pd.DataFrame(rows)
print(df.head())


         query                                       finalpassage  \
0  what is rba  Since 2007, the RBA's outstanding reputation h...   
1  what is rba  The Reserve Bank of Australia (RBA) came into ...   
2  what is rba  RBA Recognized with the 2014 Microsoft US Regi...   
3  what is rba  The inner workings of a rebuildable atomizer a...   
4  what is rba  Results-Based Accountability® (also known as R...   

                                             answers  
0  [Results-Based Accountability is a disciplined...  
1  [Results-Based Accountability is a disciplined...  
2  [Results-Based Accountability is a disciplined...  
3  [Results-Based Accountability is a disciplined...  
4  [Results-Based Accountability is a disciplined...  


In [ ]:
from sklearn.model_selection import train_test_split

train_df, valid_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42
)


In [ ]:
train_df.to_csv("train.csv", index=False)
valid_df.to_csv("valid.csv", index=False)


In [ ]:
print(pd.read_csv("train.csv").head())
print(pd.read_csv("valid.csv").head())



                                               query  \
0                           meaning of the name noma   
1  how did the civil rights act impact the civil ...   
2                      cape ann distance from boston   
3                                  what is skin rash   
4                      accounting process definition   

                                        finalpassage  \
0  : a spreading invasive gangrene chiefly of the...   
1  CIVIL RIGHTS MOVEMENT. The civil rights moveme...   
2  The total driving distance from Boston, MA to ...   
3  A rash may be localized in one part of the bod...   
4  Definition. A sequence of activities involving...   

                                             answers  
0                                           ['Fate']  
1  ['The Civil Rights Act of 1964, which ended se...  
2                                       ['30 Miles']  
3  ['It is a noticeable change in the texture or ...  
4  ['A series of activities that begins with a tr..

In [ ]:
rows = []

for item in dataset["train"]:
    query = item["query"]
    answers = item["answers"]

    passages = item["passages"]["passage_text"]
    labels = item["passages"]["is_selected"]

    for p, l in zip(passages, labels):
        rows.append({
            "query": query,
            "finalpassage": p,
            "answers": answers,
            "label": int(l)
        })


In [ ]:
df = pd.DataFrame(rows)
print(df["label"].value_counts())


label
0    587670
1     88523
Name: count, dtype: int64


In [ ]:
from sklearn.model_selection import train_test_split

train_df, valid_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)


In [ ]:
def clean_text_columns(df, query_col, text_col):
    df = df.dropna(subset=[query_col, text_col])
    df[query_col] = df[query_col].astype(str)
    df[text_col] = df[text_col].astype(str)
    return df


In [ ]:
QUERY_COL = "query"
TEXT_COL = "finalpassage"
ANSWER_COL = "answers"
LABEL_COL = "label"   # only if using labels


In [ ]:
valid_df = clean_text_columns(valid_df, QUERY_COL, TEXT_COL)


In [ ]:
# Column names
QUERY_COL = "query"
TEXT_COL = "finalpassage"
ANSWER_COL = "answers"
LABEL_COL = "label"

# Cleaning function
def clean_text_columns(df, query_col, text_col):
    df = df.dropna(subset=[query_col, text_col])
    df[query_col] = df[query_col].astype(str)
    df[text_col] = df[text_col].astype(str)
    return df


In [ ]:
print(valid_df.head())
print(valid_df.isnull().sum())


                                                    query  \
317720                                 laying a path cost   
558233  why do geologists find iceland a useful place ...   
494196                                where is steyn city   
314018                                avg temp for carmel   
555576  do piers morgan and howie mandel really dislik...   

                                             finalpassage  \
317720  1 This Old House explains how to lay a gravel ...   
558233  Scientists might find Iceland a good place to ...   
494196  WATCH: Official opening of Steyn City. INSURAN...   
314018  The highest average temperature in Carmel is J...   
555576  So I think it’s just a little competition betw...   

                                                  answers  label  
317720  [$100 -$1,000 for gravel pathway.$6 and $12 pe...      0  
558233  [Because of large amounts of lava extrusion, t...      0  
494196  [On the edge of Sandton, connecting Fourways t...      0 

In [ ]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
def rerank(df):
    pairs = list(zip(df[QUERY_COL], df[TEXT_COL]))
    scores = reranker.predict(pairs)

    df = df.copy()
    df["rerank_score"] = scores

    return df.sort_values("rerank_score", ascending=False)


In [ ]:
from sentence_transformers import CrossEncoder



In [ ]:
reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2",
    device="cuda"
)


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
import pandas as pd
from sentence_transformers import CrossEncoder

QUERY_COL = "query"
TEXT_COL = "finalpassage"
ANSWER_COL = "answers"
LABEL_COL = "label"


In [ ]:
reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2",
    device="cuda"
)


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
def rerank(df):
    pairs = list(zip(df[QUERY_COL], df[TEXT_COL]))
    scores = reranker.predict(pairs, batch_size=32)

    df = df.copy()
    df["rerank_score"] = scores
    return df.sort_values("rerank_score", ascending=False)


In [ ]:
reranked_valid = rerank(valid_df)


In [ ]:
print("Reranking completed!")
print(reranked_valid.shape)


Reranking completed!
(135239, 5)


In [ ]:
assert "reranked_valid" in globals(), "Please run reranked_valid = rerank(valid_df) first"
print("reranked_valid shape:", reranked_valid.shape)


reranked_valid shape: (135239, 5)


In [ ]:
reranked_valid[[QUERY_COL, TEXT_COL, "rerank_score"]].head(10)


,query,finalpassage,rerank_score
322502,what is a vertebral artery dissection,"Vertebral artery dissection (abbreviated VAD, ...",11.640962
131588,professional golfer average salary,Average Professional Golfer Salaries. The aver...,11.603867
479763,what is quality assurance,Quality assurance (QA) is a process-centered a...,11.602543
477369,what is soy sauce,Soy sauce (also called soya sauce) is a condim...,11.602080
199498,what is KNOWLEDGE management,Knowledge management is the name of a concept ...,11.601532
492461,what is enabling technology,"From Wikipedia, the free encyclopedia. An enab...",11.580256
654249,what is c-reactive protein,C-reactive protein (CRP) is an annular (ring-s...,11.577459
653856,what is an indemnity agreement,What Is An Indemnity Agreement?  An indemnity...,11.573626
457724,what is a vinaigrette,A vinaigrette (/vɪnəˈɡrɛt/ vin-ə-GRET) is a cu...,11.573234
46242,average salary for rural carrier associate,Average Rural Carrier Associate Salaries. The ...,11.569812


In [ ]:
reranked_by_query = (
    reranked_valid
    .groupby(QUERY_COL, group_keys=False)
    .apply(lambda x: x.sort_values("rerank_score", ascending=False))
)


/tmp/ipython-input-3003154688.py:4: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sort_values("rerank_score", ascending=False))


In [ ]:
TOP_K = 5
topk_results = reranked_by_query.groupby(QUERY_COL).head(TOP_K)



In [ ]:
sample_query = topk_results[QUERY_COL].iloc[0]

topk_results[topk_results[QUERY_COL] == sample_query][
    [TEXT_COL, "rerank_score"]
]


,finalpassage,rerank_score
322502,"Vertebral artery dissection (abbreviated VAD, ...",11.640962
322500,Vertebral artery dissection. Dr Henry Knipe ◉ ...,10.236000
322496,Vertebral artery dissection (VAD) is a relativ...,9.759173


In [ ]:
HAS_LABELS = LABEL_COL in topk_results.columns
print("Labels available:", HAS_LABELS)


Labels available: True


In [ ]:
if HAS_LABELS:
    precision_k = (
        topk_results
        .groupby(QUERY_COL)[LABEL_COL]
        .mean()
        .mean()
    )
    print(f"Precision@{TOP_K}:", precision_k)


Precision@5: 0.13423756167963646


In [ ]:
def reciprocal_rank(df):
    for idx, row in enumerate(df.itertuples(), start=1):
        if getattr(row, LABEL_COL, 0) == 1:
            return 1 / idx
    return 0


In [ ]:
if HAS_LABELS:
    mrr = (
        reranked_by_query
        .groupby(QUERY_COL)
        .apply(reciprocal_rank)
        .mean()
    )
    print("MRR:", mrr)


MRR: 0.2246649238947817


/tmp/ipython-input-2567263654.py:5: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(reciprocal_rank)


In [ ]:
import numpy as np

def ndcg_at_k(df, k):
    rels = df[LABEL_COL].tolist()
    dcg = sum(
        rel / np.log2(i + 2)
        for i, rel in enumerate(rels[:k])
    )
    ideal = sum(
        rel / np.log2(i + 2)
        for i, rel in enumerate(sorted(rels, reverse=True)[:k])
    )
    return dcg / ideal if ideal > 0 else 0


In [ ]:
if HAS_LABELS:
    ndcg = (
        reranked_by_query
        .groupby(QUERY_COL)
        .apply(lambda x: ndcg_at_k(x, TOP_K))
        .mean()
    )
    print(f"NDCG@{TOP_K}:", ndcg)


NDCG@5: 0.23192898229961065


/tmp/ipython-input-3077513138.py:5: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: ndcg_at_k(x, TOP_K))


In [ ]:
topk_results.to_csv("final_reranked_results.csv", index=False)
print("Saved: final_reranked_results.csv")


Saved: final_reranked_results.csv


In [ ]:
print("===== FINAL SUMMARY =====")
print("Total pairs:", len(reranked_valid))
print("Queries:", reranked_valid[QUERY_COL].nunique())
print("Average rerank score:", reranked_valid["rerank_score"].mean())

if HAS_LABELS:
    print(f"Precision@{TOP_K}:", precision_k)
    print("MRR:", mrr)
    print(f"NDCG@{TOP_K}:", ndcg)


===== FINAL SUMMARY =====
Total pairs: 135239
Queries: 68364
Average rerank score: 4.592267
Precision@5: 0.13423756167963646
MRR: 0.2246649238947817
NDCG@5: 0.23192898229961065
